# Benchmark Analysis: Local Agent MCP vs Direct Inference

Compares two conditions from a scripted benchmark session:
- **with_agent** — the LLM delegates code generation to the local coder MCP server
- **without_agent** — the LLM generates all code directly in its output tokens

Set `SESSION_ID` below to pin a specific session, or leave as `None` to load the latest.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

warnings.filterwarnings('ignore')

SESSION_ID = None  # e.g. '20260622_143022' — None = latest
LOGS_DIR = Path('logs')
COLORS = {'with_agent': '#4C72B0', 'without_agent': '#DD8452'}
RUN_ORDER = ['with_agent', 'without_agent']
RUN_LABELS = {'with_agent': 'With Agent', 'without_agent': 'Without Agent'}

In [ ]:
# ── Load all benchmark JSONL files ────────────────────────────────────────────
records = []
for path in LOGS_DIR.glob('benchmark_*.jsonl'):
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

if not records:
    raise FileNotFoundError(f'No benchmark_*.jsonl files found in {LOGS_DIR.resolve()}')

df_all = pd.DataFrame(records)

# Select session
if SESSION_ID is None:
    SESSION_ID = df_all['session_id'].max()
    print(f'Auto-selected latest session: {SESSION_ID}')

df = df_all[df_all['session_id'] == SESSION_ID].copy()
print(f'Session {SESSION_ID}: {len(df)} turns')
print(df.groupby('run_label').size().to_string())

In [ ]:
# ── Explode calls column into per-call DataFrame ──────────────────────────────
df_calls = df.explode('calls').reset_index(drop=True)
calls_expanded = df_calls['calls'].apply(pd.Series)
df_calls = pd.concat(
    [df_calls.drop(columns=['calls']), calls_expanded], axis=1
)

---
## 0 — Verdict

In [ ]:
wa = df[df['run_label'] == 'with_agent']
wo = df[df['run_label'] == 'without_agent']

out_saved   = int(wo['total_output_tokens'].sum() - wa['total_output_tokens'].sum())
in_overhead = int(wa['total_input_tokens'].sum()  - wo['total_input_tokens'].sum())
cost_wa     = wa['estimated_cost_usd'].sum()
cost_wo     = wo['estimated_cost_usd'].sum()
net_savings = cost_wo - cost_wa
lat_delta   = wa['total_latency_ms'].mean() - wo['total_latency_ms'].mean()
verdict     = 'WORTH IT ✓' if net_savings > 0 else 'NOT WORTH IT ✗'

print(f"""
╔══════════════════════════════════════════════════════╗
║  BENCHMARK VERDICT  —  Session {SESSION_ID}          
╠══════════════════════════════════════════════════════╣
║  Output tokens saved : {out_saved:>+,}
║  Input token overhead: {in_overhead:>+,}
║  Net cost savings    : ${net_savings:>+.6f}
║  Avg latency delta   : {lat_delta:>+.0f}ms ({'slower' if lat_delta > 0 else 'faster'} with local agent)
║  Verdict             : {verdict}
╚══════════════════════════════════════════════════════╝
""")

---
## 1 — Token Analysis
Output savings vs input overhead from extra API calls.

In [ ]:
summary = (
    df.groupby('run_label')
    .agg(
        total_input_tokens=('total_input_tokens', 'sum'),
        total_cached_tokens=('total_cached_tokens', 'sum'),
        total_output_tokens=('total_output_tokens', 'sum'),
        api_calls=('run_label', 'count'),
        tool_calls_made=('tool_calls_made', 'sum'),
        total_cost=('estimated_cost_usd', 'sum'),
        avg_latency_ms=('total_latency_ms', 'mean'),
        avg_tool_exec_ms=('tool_execution_time_ms', 'mean'),
        avg_api_latency_ms=('api_latency_ms', 'mean'),
    )
    .reindex(RUN_ORDER, fill_value=0)
)
summary['non_cached_input'] = summary['total_input_tokens'] - summary['total_cached_tokens']
summary['cache_hit_rate']   = summary['total_cached_tokens'] / summary['total_input_tokens'].clip(lower=1)
summary

In [ ]:
def bar_with_labels(ax, labels, values, colors, title, ylabel, fmt='{:,.0f}'):
    bars = ax.bar(labels, values, color=colors, width=0.5, edgecolor='white')
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', length=0)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            fmt.format(val),
            ha='center', va='bottom', fontsize=9, fontweight='bold',
        )

labels   = [RUN_LABELS[r] for r in RUN_ORDER]
colors   = [COLORS[r] for r in RUN_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Token Analysis', fontsize=13, fontweight='bold')

# Output tokens
bar_with_labels(axes[0], labels, summary['total_output_tokens'].tolist(), colors,
                'Total Output Tokens', 'Tokens')

# Stacked input: non-cached + cached
ax = axes[1]
non_cached = summary['non_cached_input'].tolist()
cached     = summary['total_cached_tokens'].tolist()
ax.bar(labels, non_cached, color=colors, width=0.5, label='Non-cached', edgecolor='white')
ax.bar(labels, cached, bottom=non_cached, color=colors, width=0.5,
       alpha=0.4, label='Cached', edgecolor='white', hatch='//')
ax.set_title('Total Input Tokens (Cached vs Non-cached)', fontweight='bold', pad=10)
ax.set_ylabel('Tokens')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', length=0)

plt.tight_layout()
plt.savefig('charts/01_token_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2 — Cost Analysis

In [ ]:
# Aggregate cost
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Cost Analysis', fontsize=13, fontweight='bold')

bar_with_labels(axes[0], labels, summary['total_cost'].tolist(), colors,
                'Total Cost (USD)', 'USD', fmt='${:.5f}')

# Per-prompt cost
per_prompt = (
    df.pivot_table(index='prompt', columns='run_label', values='estimated_cost_usd', aggfunc='sum')
    .reindex(columns=RUN_ORDER, fill_value=0)
)
per_prompt.index = [p[:40] + '…' if len(p) > 40 else p for p in per_prompt.index]
per_prompt.rename(columns=RUN_LABELS).plot(
    kind='bar', ax=axes[1], color=colors, width=0.6, edgecolor='white'
)
axes[1].set_title('Cost per Prompt (USD)', fontweight='bold', pad=10)
axes[1].set_ylabel('USD')
axes[1].tick_params(axis='x', rotation=30)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('charts/02_cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3 — Time Analysis
**Left**: stacked decomposition — API time vs local tool execution time.
**Right**: per-prompt latency delta (`with_agent − without_agent`). Negative = local agent was faster.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Time Analysis', fontsize=13, fontweight='bold')

# Stacked: API latency + tool execution
ax = axes[0]
api_lat  = summary['avg_api_latency_ms'].tolist()
tool_lat = summary['avg_tool_exec_ms'].tolist()
ax.bar(labels, api_lat,  color=colors, width=0.5, label='API latency', edgecolor='white')
ax.bar(labels, tool_lat, bottom=api_lat, color=colors, width=0.5,
       alpha=0.45, label='Tool exec', edgecolor='white', hatch='\\\\')
ax.set_title('Avg Turn Latency Decomposition', fontweight='bold', pad=10)
ax.set_ylabel('ms')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='x', length=0)

# Per-prompt delta
lat_pivot = (
    df.pivot_table(index='prompt', columns='run_label', values='total_latency_ms', aggfunc='mean')
    .reindex(columns=RUN_ORDER, fill_value=0)
)
lat_pivot['delta'] = lat_pivot['with_agent'] - lat_pivot['without_agent']
short_prompts = [p[:35] + '…' if len(p) > 35 else p for p in lat_pivot.index]
delta_colors = ['#e04040' if d > 0 else '#40a040' for d in lat_pivot['delta']]
axes[1].barh(short_prompts, lat_pivot['delta'], color=delta_colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Latency Delta per Prompt\n(+ms = local agent slower)', fontweight='bold', pad=10)
axes[1].set_xlabel('ms')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('charts/03_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 — Cache Efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
cache_rates = (summary['cache_hit_rate'] * 100).tolist()
bar_with_labels(ax, labels, cache_rates, colors,
                'Cache Hit Rate (cached / total input tokens)', '%', fmt='{:.1f}%')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('charts/04_cache_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5 — Per-Prompt ROI Ranking
Prompts sorted by net cost savings (`without_agent − with_agent`). Positive = local agent saved money.

In [ ]:
cost_pivot = df.pivot_table(
    index='prompt', columns='run_label', values='estimated_cost_usd', aggfunc='sum'
).reindex(columns=RUN_ORDER, fill_value=0)

cost_pivot['net_savings'] = cost_pivot['without_agent'] - cost_pivot['with_agent']
cost_pivot = cost_pivot.sort_values('net_savings', ascending=True)
short_prompts = [p[:50] + '…' if len(p) > 50 else p for p in cost_pivot.index]

roi_colors = ['#40a040' if s > 0 else '#e04040' for s in cost_pivot['net_savings']]
fig, ax = plt.subplots(figsize=(10, max(4, len(short_prompts) * 0.7)))
ax.barh(short_prompts, cost_pivot['net_savings'], color=roi_colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Per-Prompt Net Cost Savings\n(+USD = local agent cheaper)', fontweight='bold', pad=10)
ax.set_xlabel('USD saved')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/05_roi_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nROI ranking (high to low):')
for prompt, row in cost_pivot.sort_values('net_savings', ascending=False).iterrows():
    sign = '✓' if row['net_savings'] > 0 else '✗'
    print(f"  {sign}  ${row['net_savings']:>+.6f}  {prompt[:60]}")

---
## 6 — Calls-per-Turn Distribution
How many API calls did each turn require in the `with_agent` run?

In [ ]:
wa_calls = wa['calls'].apply(len)
fig, ax = plt.subplots(figsize=(6, 4))
max_calls = int(wa_calls.max()) if len(wa_calls) else 1
bins = range(1, max_calls + 2)
ax.hist(wa_calls, bins=bins, align='left', color=COLORS['with_agent'],
        edgecolor='white', rwidth=0.7)
ax.set_title('Calls-per-Turn Distribution (with_agent)', fontweight='bold', pad=10)
ax.set_xlabel('API calls per turn')
ax.set_ylabel('Number of turns')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('charts/06_calls_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean calls/turn: {wa_calls.mean():.2f}  |  Max: {wa_calls.max()}')

---
## 7 — Per-Call Token & Latency Breakdown

Each API call within a turn is indexed by `call_index` (0 = first call, 1 = after first tool result, etc.).
- `without_agent` turns have only call 0.
- `with_agent` turns have call 0 (tool request) + call 1+ (processing tool result → final answer).

This shows exactly where tokens accumulate across the agentic loop.

In [ ]:
# Average tokens and latency per call_index, per run
per_call = (
    df_calls
    .groupby(['run_label', 'call_index'])
    .agg(
        avg_input=('input_tokens', 'mean'),
        avg_output=('output_tokens', 'mean'),
        avg_cached=('cached_tokens', 'mean'),
        avg_latency=('latency_ms', 'mean'),
        n=('input_tokens', 'count'),
    )
    .reset_index()
)
per_call['call_label'] = per_call.apply(
    lambda r: f"{RUN_LABELS[r['run_label']]}\ncall {int(r['call_index'])}", axis=1
)
per_call['bar_color'] = per_call['run_label'].map(COLORS)
per_call['fade'] = per_call['call_index'].apply(lambda i: 0.65 if i > 0 else 1.0)

print(per_call[['run_label', 'call_index', 'avg_input', 'avg_output', 'avg_cached', 'avg_latency', 'n']]
      .round(1).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Per-Call Breakdown (avg across turns)', fontsize=13, fontweight='bold')

def call_bar(ax, values, title, ylabel):
    bars = ax.bar(per_call['call_label'], values,
                  color=per_call['bar_color'], width=0.55, edgecolor='white')
    for bar, fade in zip(bars, per_call['fade']):
        bar.set_alpha(fade)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_ylabel(ylabel)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', labelsize=8)
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f'{int(val):,}',
            ha='center', va='bottom', fontsize=8, fontweight='bold',
        )

call_bar(axes[0], per_call['avg_input'].tolist(),   'Avg Input Tokens / Call',  'Tokens')
call_bar(axes[1], per_call['avg_output'].tolist(),  'Avg Output Tokens / Call', 'Tokens')
call_bar(axes[2], per_call['avg_latency'].tolist(), 'Avg Latency / Call',       'ms')

import matplotlib.patches as mpatches
fig.legend(
    handles=[mpatches.Patch(color=COLORS[r], label=RUN_LABELS[r]) for r in RUN_ORDER],
    loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.06), frameon=False,
)
plt.tight_layout()
plt.savefig('charts/07_per_call_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Stacked token composition per call — with_agent vs without_agent side by side
wa_calls_df = df_calls[df_calls['run_label'] == 'with_agent'].copy()
wo_calls_df = df_calls[df_calls['run_label'] == 'without_agent'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Token Composition per Call (avg) — input growth across the agentic loop',
             fontsize=12, fontweight='bold')

def stacked_call_chart(ax, df_run, run_label):
    grp = df_run.groupby('call_index')[['input_tokens', 'output_tokens', 'cached_tokens']].mean()
    grp['non_cached_input'] = grp['input_tokens'] - grp['cached_tokens']
    x = [f'Call {i}' for i in grp.index]
    c = COLORS[run_label]
    b0 = grp['non_cached_input']
    b1 = grp['cached_tokens']
    b2 = grp['output_tokens']
    ax.bar(x, b0,            color=c, alpha=1.0,  width=0.5, edgecolor='white', label='Non-cached input')
    ax.bar(x, b1, bottom=b0, color=c, alpha=0.35, width=0.5, edgecolor='white', hatch='//',  label='Cached input')
    ax.bar(x, b2, bottom=b0+b1, color=c, alpha=0.7, width=0.5, edgecolor='white', hatch='\\\\', label='Output')
    ax.set_title(RUN_LABELS[run_label], fontweight='bold', pad=8)
    ax.set_ylabel('Avg tokens')
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.legend(fontsize=8, frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

stacked_call_chart(axes[0], wa_calls_df, 'with_agent')
stacked_call_chart(axes[1], wo_calls_df, 'without_agent')

plt.tight_layout()
plt.savefig('charts/07b_per_call_stacked.png', dpi=150, bbox_inches='tight')
plt.show()